<a href="https://colab.research.google.com/github/MarcinPlodzien/deep-learning-for-scientific-discovery/blob/main/notebooks/lecture-06/generative-models.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Lecture 6: Learning a Probability Distribution from Data

*Deep Learning for Scientific Discovery, Jagiellonian University. Author: Marcin Płodzień.*

The data $\mathbf x_1,\dots,\mathbf x_n$ we collect are **samples** from an unknown probability distribution $p(\mathbf x)$: the distribution of handwritten digits, of cell images, of spin configurations of a magnet at temperature $T$. A **generative model** learns an approximation $p_\theta(\mathbf x)$ of it from the samples, and can then **draw new samples**: new instances that were never in the data set but look as if they could have been.

This notebook goes through the main ideas, each with a small experiment that runs on a laptop CPU:

0. the probability toolkit: distributions, expectations, the Gaussian, joint/conditional/marginal, KL divergence;
1. probability and likelihood;
2. maximum likelihood: fitting a distribution to points in 2D;
3. why an ordinary autoencoder cannot generate;
4. the variational autoencoder (VAE): new digits from random latent vectors;
5. the conditional VAE: a digit we choose;
6. the conditional VAE on blood-cell images: a cell type we choose;
7. generative adversarial networks (GANs), the best discriminator, the Wasserstein distance, and the difficulty of training;
8. diffusion models: the forward process in 1D, the learned direction field on a 2D example, and MNIST;
9. the 2D Ising model, where the true distribution is known: a conditional VAE that generates spin configurations at a chosen temperature, checked against Monte Carlo.

All random generators are seeded, so the numbers agree with the slides. Each part ends with a **Try it** exercise.

## Setup

In [1]:
import os, json, time
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
torch.set_num_threads(4)
import os
OUT = "figures"; os.makedirs(OUT, exist_ok=True); os.makedirs(OUT, exist_ok=True)      # figures of the slides
RES = "results"; os.makedirs(RES, exist_ok=True)
blue, orange, navy = "#005CA7", "#EB6709", "#0A2F5C"
cmap = plt.get_cmap("tab10")
nums = {}                                                        # numbers quoted on the slides
def clean(ax):
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
def seed(k):
    torch.manual_seed(k); return np.random.default_rng(k)

## 0. The probability toolkit

Everything in this lecture is written in the language of probability. This part collects the few objects we need, each with a small computation: discrete and continuous distributions, expectations as averages over samples, the Gaussian, joint / conditional / marginal distributions, the Kullback–Leibler divergence, and its link to maximum likelihood.

### Discrete and continuous distributions

* **Discrete** $x$ (a die, a class label, a spin): $p(x)$ is a probability, $0\le p(x)\le 1$, and $\sum_x p(x)=1$.
* **Continuous** $x$ (a pixel intensity, a temperature, a coordinate): $p(x)$ is a **density**. Only areas are probabilities, $P(a\le x\le b)=\int_a^b p(x)\,dx$, and $\int p(x)\,dx=1$. A density can be larger than 1.

In [2]:
from scipy.stats import norm
xs = np.linspace(-4, 4, 1601)
nums["gauss_P_within_1sigma"] = float(norm.cdf(1) - norm.cdf(-1))
nums["gauss_peak_sigma02"] = float(norm.pdf(0, 0, 0.2))
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
axs[0].bar(range(1, 7), [1 / 6] * 6, color=blue, width=0.6)
axs[0].set_xlabel("outcome $x$ of a die"); axs[0].set_ylabel("$p(x)$")
axs[0].set_title(r"discrete: $p(x) = 1/6$, $\sum_x p(x) = 1$", fontsize=11)
axs[1].plot(xs, norm.pdf(xs, 0, 0.2), color=navy, lw=1.4, ls="--", label=r"$\mathcal{N}(0, 0.2^2)$: peak 1.99")
axs[1].plot(xs, norm.pdf(xs), color=blue, lw=2.2, label=r"$\mathcal{N}(0, 1)$")
m = (xs >= -1) & (xs <= 1); axs[1].fill_between(xs[m], norm.pdf(xs[m]), color=orange, alpha=0.55, label=r"$P(-1 \leq x \leq 1) = 0.683$")
axs[1].set_xlabel("$x$"); axs[1].set_ylabel("density $p(x)$"); axs[1].set_ylim(0, 2.1)
axs[1].set_title(r"continuous: probability = area, $\int p(x)\,dx = 1$", fontsize=11); axs[1].legend(frameon=False, fontsize=9)
for a in axs: clean(a)
plt.tight_layout(); plt.savefig(f"{OUT}/gen_discrete_continuous.png", dpi=170, bbox_inches="tight"); plt.show()
print(f"P(-1 <= x <= 1) for N(0,1): {nums['gauss_P_within_1sigma']:.3f};  peak of N(0, 0.2^2): {nums['gauss_peak_sigma02']:.2f} > 1")

P(-1 <= x <= 1) for N(0,1): 0.683;  peak of N(0, 0.2^2): 1.99 > 1


/tmp/ipykernel_3869297/628720634.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_discrete_continuous.png", dpi=170, bbox_inches="tight"); plt.show()


The area under each curve is 1; the narrow Gaussian has a peak of about 2, so a density is not a probability. For an image, $\mathbf x\in\mathbb R^{784}$ and $p(\mathbf x)$ is a density in 784 dimensions.

### Expectation, and averages over samples

The **expectation** of a function $f$ under $p$ is its average weighted by the probabilities,
$$\mathbb E_{x\sim p}[f(x)]=\sum_x p(x)\,f(x)\quad\text{or}\quad\int p(x)\,f(x)\,dx .$$
We almost never know $p$, but we have samples $x_1,\dots,x_n$ from it. Then
$$\mathbb E_{x\sim p}[f(x)]\approx\frac1n\sum_{i=1}^n f(x_i),$$
with an error that shrinks like $1/\sqrt n$. Every loss in this lecture is such an average, computed on a mini-batch.

Example: a mixture of two Gaussians, $p(x)=\tfrac12\mathcal N(x;-2,0.6^2)+\tfrac12\mathcal N(x;1.5,1)$, whose mean is exactly $\mathbb E[x]=\tfrac12(-2)+\tfrac12(1.5)=-0.25$.

In [3]:
def p_mix(x):                                               # the density of the example
    return 0.5 * norm.pdf(x, -2, 0.6) + 0.5 * norm.pdf(x, 1.5, 1.0)
def sample_mix(n, rng):
    k = rng.random(n) < 0.5
    return np.where(k, rng.normal(-2, 0.6, n), rng.normal(1.5, 1.0, n))
rng = seed(20)
fig, axs = plt.subplots(1, 3, figsize=(11, 2.9), sharey=True)
est = {}
for a, n in zip(axs, [10, 100, 10000]):
    s = sample_mix(n, rng); est[n] = float(s.mean())
    a.hist(s, bins=np.linspace(-4.5, 5, 40), density=True, color=blue, alpha=0.6)
    a.plot(xs * 1.2, p_mix(xs * 1.2), color=orange, lw=2)
    a.set_title(f"n = {n}: sample mean {s.mean():+.2f}", fontsize=11); a.set_xlabel("$x$"); clean(a)
axs[0].set_ylabel("density")
nums["mix_mean_estimates"] = {str(k): round(v, 3) for k, v in est.items()}
plt.tight_layout(); plt.savefig(f"{OUT}/gen_samples_expectation.png", dpi=170, bbox_inches="tight"); plt.show()
print("exact E[x] = -0.25; estimates from n samples:", nums["mix_mean_estimates"])

exact E[x] = -0.25; estimates from n samples: {'10': -1.142, '100': -0.591, '10000': -0.282}


/tmp/ipykernel_3869297/2944942503.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_samples_expectation.png", dpi=170, bbox_inches="tight"); plt.show()


With 10 samples the histogram hardly resembles $p$ and the mean is off; with 10 000 both are close. A generative model only ever sees such samples.

### The Gaussian distribution

In one dimension, with mean $\mu$ and standard deviation $\sigma$,
$$\mathcal N(x;\mu,\sigma^2)=\frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\Big(-\frac{(x-\mu)^2}{2\sigma^2}\Big).$$
In $d$ dimensions with independent coordinates, $\mathcal N(\mathbf z;\boldsymbol\mu,\mathrm{diag}\,\boldsymbol\sigma^2)=\prod_{k=1}^d\mathcal N(z_k;\mu_k,\sigma_k^2)$; the **standard** Gaussian is $\mathcal N(0,I)$ ($\mu_k=0$, $\sigma_k=1$). To draw from $\mathcal N(\boldsymbol\mu,\mathrm{diag}\,\boldsymbol\sigma^2)$, draw $\boldsymbol\varepsilon\sim\mathcal N(0,I)$ and compute $\mathbf z=\boldsymbol\mu+\boldsymbol\sigma\odot\boldsymbol\varepsilon$ ($\odot$: element-wise product). This shift-and-stretch is the reparametrisation trick of the VAE.

In [4]:
rng = seed(21)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
for (mu, sg), col in zip([(0, 1), (2, 0.5), (-1.5, 1.5)], [blue, orange, navy]):
    axs[0].plot(xs * 1.5, norm.pdf(xs * 1.5, mu, sg), color=col, lw=2, label=rf"$\mu = {mu}$, $\sigma = {sg}$")
axs[0].legend(frameon=False, fontsize=9); axs[0].set_xlabel("$x$"); axs[0].set_title(r"$\mathcal{N}(x; \mu, \sigma^2)$", fontsize=11)
eps = rng.standard_normal((1500, 2)); mu2, sg2 = np.array([3.0, 1.0]), np.array([0.5, 1.5])
z = mu2 + sg2 * eps
axs[1].scatter(*eps.T, s=2, color="grey", label=r"$\boldsymbol{\varepsilon} \sim \mathcal{N}(0, I)$")
axs[1].scatter(*z.T, s=2, color=orange, label=r"$\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\varepsilon}$")
axs[1].set_aspect("equal"); axs[1].legend(frameon=False, fontsize=9, loc="lower left", markerscale=4)
axs[1].set_title(r"$\boldsymbol{\mu} = (3, 1)$, $\boldsymbol{\sigma} = (0.5, 1.5)$", fontsize=11)
for a in axs: clean(a)
plt.tight_layout(); plt.savefig(f"{OUT}/gen_gaussian.png", dpi=170, bbox_inches="tight"); plt.show()
print("sample mean of z:", z.mean(0).round(2), " sample std of z:", z.std(0).round(2))

sample mean of z: [3.02 1.05]  sample std of z: [0.49 1.47]


/tmp/ipykernel_3869297/1138759995.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_gaussian.png", dpi=170, bbox_inches="tight"); plt.show()


### Joint, conditional and marginal distributions

Two variables, an observed $x$ and a hidden (latent) $z$. Their **joint** distribution $p(x,z)$ gives the probability of both together. From it:

* **marginal**: $p(x)=\int p(x,z)\,dz$, the distribution of $x$ alone, whatever $z$ is;
* **conditional**: $p(x\,|\,z)=p(x,z)/p(z)$, the distribution of $x$ once $z$ is known; hence the **product rule** $p(x,z)=p(x\,|\,z)\,p(z)$;
* **Bayes' rule**: $p(z\,|\,x)=p(x\,|\,z)\,p(z)/p(x)$, what an observed $x$ tells about the hidden $z$ (the **posterior**).

Example (a latent-variable model): $z\sim\mathcal N(0,1)$ and, given $z$, $x\sim\mathcal N\big(2\tanh(2z),\,0.35^2\big)$. The marginal $p(x)$ is not Gaussian, although both ingredients are.

In [5]:
zg = np.linspace(-3, 3, 601); xg = np.linspace(-3.5, 3.5, 701); dz, dx = zg[1] - zg[0], xg[1] - xg[0]
fz = lambda z: 2 * np.tanh(2 * z); s_x = 0.35
joint = norm.pdf(zg)[None, :] * norm.pdf(xg[:, None], fz(zg)[None, :], s_x)     # rows: x, columns: z
px = joint.sum(1) * dz                                                           # marginal p(x)
z0, x0 = -0.5, 1.0
post = joint[np.argmin(abs(xg - x0))] / px[np.argmin(abs(xg - x0))]                # p(z | x0), Bayes
nums["post_mean"] = float((zg * post).sum() * dz); nums["post_std"] = float(np.sqrt(((zg - nums["post_mean"]) ** 2 * post).sum() * dz))
fig, axs = plt.subplots(1, 3, figsize=(12, 3.4))
axs[0].imshow(joint, origin="lower", extent=[zg[0], zg[-1], xg[0], xg[-1]], aspect="auto", cmap="Blues")
axs[0].axvline(z0, color=orange, lw=1.5); axs[0].axhline(x0, color=navy, lw=1.5, ls="--")
axs[0].set_xlabel("latent $z$"); axs[0].set_ylabel("observed $x$"); axs[0].set_title("joint $p(x, z)$", fontsize=11)
axs[1].plot(xg, px, color=blue, lw=2.2, label="marginal $p(x)$")
axs[1].plot(xg, norm.pdf(xg, fz(z0), s_x), color=orange, lw=2, label=f"conditional $p(x\\,|\\,z = {z0})$")
axs[1].set_xlabel("$x$"); axs[1].legend(frameon=False, fontsize=9); axs[1].set_title("marginal and conditional", fontsize=11)
axs[2].plot(zg, norm.pdf(zg), color="grey", lw=1.5, ls=":", label="prior $p(z)$")
axs[2].plot(zg, post, color=navy, lw=2.2, label=f"posterior $p(z\\,|\\,x = {x0})$")
axs[2].set_xlabel("$z$"); axs[2].legend(frameon=False, fontsize=9, loc="upper left"); axs[2].set_title("Bayes' rule", fontsize=11)
for a in axs[1:]: clean(a)
plt.tight_layout(); plt.savefig(f"{OUT}/gen_joint_conditional.png", dpi=170, bbox_inches="tight"); plt.show()
print(f"integral of p(x): {px.sum() * dx:.4f};  posterior p(z | x = {x0}): mean {nums['post_mean']:.3f}, std {nums['post_std']:.3f}")

integral of p(x): 0.9973;  posterior p(z | x = 1.0): mean 0.360, std 0.245


/tmp/ipykernel_3869297/3182169493.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_joint_conditional.png", dpi=170, bbox_inches="tight"); plt.show()


The marginal $p(x)$ has two peaks: most $z$ are mapped by $\tanh$ close to $\pm2$. Observing $x=1$ narrows the possible $z$ from the broad prior (standard deviation 1) to a narrow posterior (printed above: mean about 0.36, standard deviation about 0.25), close to $z=0.27$ where $2\tanh(2z)=1$. A VAE needs exactly this posterior, and its **encoder** learns an approximation of it.

### The Kullback–Leibler divergence and maximum likelihood

How far is a model $q$ from the true distribution $p$? The **KL divergence**
$$\mathrm{KL}(p\,\|\,q)=\mathbb E_{x\sim p}\Big[\log\frac{p(x)}{q(x)}\Big]=\int p(x)\log\frac{p(x)}{q(x)}\,dx$$
is $\ge0$, equal to 0 only when $q=p$, and not symmetric. It is the average *extra surprise* when data from $p$ are described by $q$.

Split the logarithm: $\mathrm{KL}(p\,\|\,p_\theta)=\mathbb E_p[\log p]-\mathbb E_p[\log p_\theta]$. The first term does not depend on $\theta$, and the second is estimated by the mean log-likelihood $\frac1n\sum_i\log p_\theta(x_i)$. Hence
$$\text{maximising the log-likelihood}\;\Longleftrightarrow\;\text{minimising }\mathrm{KL}(p\,\|\,p_\theta).$$
Check: fit one Gaussian $\mathcal N(\mu,\sigma^2)$ to the two-peak mixture above. The KL is smallest at $\mu=\mathbb E_p[x]=-0.25$, and maximum likelihood on samples (the sample mean) lands there too.

In [6]:
xk = np.linspace(-8, 8, 4001); dxk = xk[1] - xk[0]; pk = p_mix(xk)
var_p = float(((xk + 0.25) ** 2 * pk).sum() * dxk)
def kl(p, q): return float((p * (np.log(p + 1e-300) - np.log(q + 1e-300))).sum() * dxk)
mus = np.linspace(-2.5, 2.0, 181)
kls = [kl(pk, norm.pdf(xk, m_, np.sqrt(var_p))) for m_ in mus]
rng = seed(22); smp = sample_mix(10000, rng)
q_star = norm.pdf(xk, smp.mean(), smp.std())
nums["kl_mu_min"] = float(mus[np.argmin(kls)]); nums["kl_ml_mean"] = float(smp.mean()); nums["kl_ml_std"] = float(smp.std())
nums["kl_p_qstar"] = kl(pk, q_star); nums["kl_qstar_p"] = kl(q_star, pk)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
axs[0].plot(xk, pk, color=blue, lw=2.2, label="data $p(x)$")
axs[0].plot(xk, q_star, color=orange, lw=2, label="Gaussian fitted by max. likelihood")
axs[0].set_xlim(-5, 5); axs[0].set_xlabel("$x$"); axs[0].legend(frameon=False, fontsize=9)
axs[0].set_title(f"KL(p || q) = {nums['kl_p_qstar']:.2f},  KL(q || p) = {nums['kl_qstar_p']:.2f}", fontsize=11)
axs[1].plot(mus, kls, color=navy, lw=2); axs[1].axvline(-0.25, color=orange, ls="--", lw=1.5, label=r"$\mathbb{E}_p[x] = -0.25$")
axs[1].axvline(smp.mean(), color=blue, ls=":", lw=2, label=f"max. likelihood: {smp.mean():.3f}")
axs[1].set_xlabel(r"mean $\mu$ of the Gaussian $q$"); axs[1].set_ylabel(r"KL$(p \,\|\, q)$"); axs[1].legend(frameon=False, fontsize=9)
axs[1].set_title(r"KL$(p \,\|\, \mathcal{N}(\mu, \sigma_p^2))$ against $\mu$", fontsize=11)
for a in axs: clean(a)
plt.tight_layout(); plt.savefig(f"{OUT}/gen_kl_ml.png", dpi=170, bbox_inches="tight"); plt.show()
print(f"KL minimum at mu = {nums['kl_mu_min']:.3f}; max-likelihood mean {smp.mean():.3f}, std {smp.std():.3f} (exact {np.sqrt(var_p):.3f})")
print(f"KL(p || q*) = {nums['kl_p_qstar']:.3f}, KL(q* || p) = {nums['kl_qstar_p']:.3f}: not symmetric")

KL minimum at mu = -0.250; max-likelihood mean -0.211, std 1.929 (exact 1.935)
KL(p || q*) = 0.262, KL(q* || p) = 0.454: not symmetric


/tmp/ipykernel_3869297/283658985.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_kl_ml.png", dpi=170, bbox_inches="tight"); plt.show()


The best single Gaussian sits between the two peaks and covers both: it cannot be close to $p$, and the KL measures how far it remains. The two directions of the KL give different numbers.

## 1. Probability and likelihood

A model $p(x\,|\,\theta)$ gives the probability of data $x$ when the parameters are $\theta$. The same formula is read in two ways:

* **probability**: $\theta$ fixed, $x$ varies. Which data should we expect? Summed over all outcomes $x$ it gives 1.
* **likelihood**: the observed data $x$ fixed, $\theta$ varies, $L(\theta)=p(x_{\text{obs}}\,|\,\theta)$. Which parameters make the observed data most plausible? It is a function of $\theta$, not a probability distribution over $\theta$.

Example: a coin with probability $\theta$ of heads, tossed 10 times. The number of heads $k$ follows $p(k\,|\,\theta)=\binom{10}{k}\theta^k(1-\theta)^{10-k}$. We observed $k=7$.

In [7]:
from math import comb
ks = np.arange(11); th = np.linspace(0, 1, 401)
p_k = np.array([comb(10, k) * 0.5 ** 10 for k in ks])
L_th = comb(10, 7) * th ** 7 * (1 - th) ** 3
nums["coin_p7_given_half"] = float(p_k[7]); nums["coin_L_at_07"] = float(comb(10, 7) * 0.7 ** 7 * 0.3 ** 3)
nums["coin_L_integral"] = float(np.trapezoid(L_th, th)) if hasattr(np, "trapezoid") else float(np.trapz(L_th, th))
fig, axs = plt.subplots(1, 2, figsize=(9.5, 3.2))
axs[0].bar(ks, p_k, color=blue); axs[0].bar([7], [p_k[7]], color=orange)
axs[0].set_xlabel("number of heads $k$"); axs[0].set_title(r"probability $p(k\,|\,\theta = 0.5)$: sums to 1 over $k$", fontsize=10)
axs[1].plot(th, L_th, color=orange, lw=2.2); axs[1].axvline(0.7, color="k", ls=":", lw=1)
axs[1].set_xlabel(r"parameter $\theta$"); axs[1].set_title(r"likelihood $L(\theta) = p(k = 7\,|\,\theta)$: maximum at $\theta = 0.7$", fontsize=10)
for a in axs: clean(a)
plt.tight_layout(); plt.savefig(f"{OUT}/gen_prob_vs_likelihood.png", dpi=170, bbox_inches="tight"); plt.show()
print(f"p(7 | 0.5) = {nums['coin_p7_given_half']:.3f};  L(0.7) = {nums['coin_L_at_07']:.3f};  integral of L over theta = {nums['coin_L_integral']:.4f} (= 1/11, not 1)")

p(7 | 0.5) = 0.117;  L(0.7) = 0.267;  integral of L over theta = 0.0909 (= 1/11, not 1)


/tmp/ipykernel_3869297/949992923.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_prob_vs_likelihood.png", dpi=170, bbox_inches="tight"); plt.show()


## 2. Maximum likelihood

Data $\mathbf x_1,\dots,\mathbf x_n$, independent samples of an unknown $p(\mathbf x)$, and a model $p_\theta(\mathbf x)$ with parameters $\theta$. The **likelihood** of the data is the probability (density) the model assigns to all of them, $L(\theta)=\prod_{i=1}^n p_\theta(\mathbf x_i)$. Maximum likelihood picks the $\theta$ that makes the observed data most probable; in practice we maximise the log-likelihood
$$\ell(\theta)=\sum_{i=1}^n \log p_\theta(\mathbf x_i).$$
(Maximising $\ell$ is the same as minimising the KL divergence between the distribution of the data and the model; the cross-entropy loss of classification is the same idea.)

Example: 1500 points in 2D drawn from three groups. We fit two models by maximum likelihood with scikit-learn: a single Gaussian, and a mixture of three Gaussians, $p_\theta(\mathbf x)=\sum_{k=1}^{3}\pi_k\,\mathcal N(\mathbf x;\boldsymbol\mu_k,\Sigma_k)$. Then we **draw new points** from each fitted model.

In [8]:
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs
X2, _ = make_blobs(1500, centers=[[-3, 0], [2, 2.5], [2.5, -2.5]], cluster_std=[0.9, 0.7, 1.1], random_state=0)
models = {"one Gaussian": GaussianMixture(1, random_state=0).fit(X2),
          "mixture of three Gaussians": GaussianMixture(3, random_state=0).fit(X2)}
xx, yy = np.meshgrid(np.linspace(-7, 7, 200), np.linspace(-6.5, 6.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
fig, axs = plt.subplots(1, 3, figsize=(12, 3.7))
axs[0].scatter(X2[:, 0], X2[:, 1], s=3, color="grey"); axs[0].set_title("data: 1500 samples", fontsize=11)
for a, (name, m) in zip(axs[1:], models.items()):
    dens = np.exp(m.score_samples(grid)).reshape(xx.shape)
    a.contourf(xx, yy, dens, levels=12, cmap="Blues")
    new, _ = m.sample(400)
    a.scatter(new[:, 0], new[:, 1], s=3, color=orange)
    nums[f"mle_loglik_{len(m.weights_)}"] = float(m.score(X2))       # mean log-likelihood per point
    a.set_title(f"{name}\nlog-likelihood per point {m.score(X2):.2f}", fontsize=10)
for a in axs: a.set_xticks([]); a.set_yticks([]); a.set_aspect("equal"); clean(a)
plt.tight_layout(); plt.savefig(f"{OUT}/gen_mle_gmm.png", dpi=170, bbox_inches="tight"); plt.show()
print("blue: fitted density; orange: 400 new samples drawn from the fitted model")

blue: fitted density; orange: 400 new samples drawn from the fitted model


/tmp/ipykernel_3869297/2519545018.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_mle_gmm.png", dpi=170, bbox_inches="tight"); plt.show()


The mixture assigns the data a higher likelihood and its samples look like the data; a single Gaussian puts probability where there is no data. For images we cannot write such a formula by hand: the next parts let neural networks do it.

**Try it.** Fit mixtures with 2, 3, 6 and 12 components and plot the log-likelihood on the data and on a fresh set of samples from `make_blobs` (another `random_state`). When does a larger model stop helping?

## 3. An Autoencoder Cannot Generate

We read the MNIST files (they are downloaded once into `data/MNIST_raw` if they are not there). Pixels are scaled to $[0,1]$; each image is a vector of 784 numbers.

An autoencoder (Lecture 5) maps an image to a latent vector $\mathbf z=E(\mathbf x)$ and back, $\hat{\mathbf x}=G(\mathbf z)$. Can its decoder $G$ produce *new* digits? We train one with $d=2$ and decode a regular grid of latent points covering the region where the training codes lie.

In [9]:
import gzip, urllib.request
RAW = "data/MNIST_raw"; os.makedirs(RAW, exist_ok=True)
LOCAL = os.path.expanduser("~/.pytorch/MNIST_data/MNIST/raw")
MIRRORS = ["https://ossci-datasets.s3.amazonaws.com/mnist/", "https://storage.googleapis.com/cvdf-datasets/mnist/"]
def mnist_file(name):
    for d in (RAW, LOCAL):
        p = os.path.join(d, name)
        if os.path.exists(p): return open(p, "rb").read()
        if os.path.exists(p + ".gz"): return gzip.open(p + ".gz").read()
    for m in MIRRORS:
        try:
            urllib.request.urlretrieve(m + name + ".gz", os.path.join(RAW, name + ".gz"))
            return gzip.open(os.path.join(RAW, name + ".gz")).read()
        except Exception: pass
    raise RuntimeError("MNIST could not be found or downloaded")
def idx(name):
    d = mnist_file(name)
    if d[3] == 3:
        n, r, c = np.frombuffer(d[4:16], ">i4"); return np.frombuffer(d[16:], np.uint8).reshape(n, r * c)
    return np.frombuffer(d[8:], np.uint8)
Xtr = idx("train-images-idx3-ubyte").astype(np.float32) / 255; ytr = idx("train-labels-idx1-ubyte")
Xte = idx("t10k-images-idx3-ubyte").astype(np.float32) / 255; yte = idx("t10k-labels-idx1-ubyte")
Xtr_t, Xte_t = torch.tensor(Xtr), torch.tensor(Xte)
print("training images:", Xtr.shape, " test images:", Xte.shape)

def mlp(sizes, last=None):
    layers = []
    for a, b in zip(sizes[:-1], sizes[1:]):
        layers += [nn.Linear(a, b), nn.ReLU()]
    layers = layers[:-1]
    if last is not None: layers.append(last)
    return nn.Sequential(*layers)

def show_grid(imgs, nrow, ncol, path, title=None, size=1.0):
    fig, axs = plt.subplots(nrow, ncol, figsize=(ncol * size, nrow * size))
    for a, im in zip(axs.ravel(), imgs):
        a.imshow(im.reshape(28, 28), cmap="gray_r", vmin=0, vmax=1); a.axis("off")
    plt.subplots_adjust(wspace=0.03, hspace=0.03)
    if title: fig.suptitle(title, fontsize=10)
    plt.savefig(path, dpi=170, bbox_inches="tight"); plt.show()

rng = seed(0)
enc_ae, dec_ae = mlp([784, 256, 64, 2]), mlp([2, 64, 256, 784], nn.Sigmoid())
opt = torch.optim.Adam(list(enc_ae.parameters()) + list(dec_ae.parameters()), 1e-3)
for ep in range(15):
    perm = torch.randperm(len(Xtr_t))
    for i in range(0, len(perm), 256):
        b = Xtr_t[perm[i:i + 256]]
        loss = ((dec_ae(enc_ae(b)) - b) ** 2).mean(); opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    Z = enc_ae(Xte_t).numpy()
lo, hi = np.percentile(Z, 1, 0), np.percentile(Z, 99, 0)
g1, g2 = np.linspace(lo[0], hi[0], 12), np.linspace(hi[1], lo[1], 12)
zg = torch.tensor([[a, b] for b in g2 for a in g1], dtype=torch.float32)
with torch.no_grad(): imgs = dec_ae(zg).numpy()
show_grid(imgs, 12, 12, f"{OUT}/gen_ae_grid.png", size=0.6)

training images: (60000, 784)  test images: (10000, 784)


/tmp/ipykernel_3869297/2771021278.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(path, dpi=170, bbox_inches="tight"); plt.show()


Some regions decode to digits, others to shapes that are no digit at all, and nothing tells us which latent points are "valid". The autoencoder learned to rebuild its training images, not a distribution we can sample. To generate we need a latent space with a **known distribution**.

## 4. The variational autoencoder (VAE)

The VAE is a latent-variable model: a latent vector $\mathbf z$ is drawn from a simple distribution $p(\mathbf z)=\mathcal N(0,I)$, and the decoder turns it into an image, $p_\theta(\mathbf x|\mathbf z)$. The distribution of the data is then
$$p_\theta(\mathbf x)=\int p_\theta(\mathbf x|\mathbf z)\,p(\mathbf z)\,d\mathbf z .$$

* **Encoder.** For an input $\mathbf x$ it outputs a Gaussian *cloud* in latent space: a mean $\boldsymbol\mu(\mathbf x)$ and a width $\boldsymbol\sigma(\mathbf x)$, one pair for each of the $d$ latent coordinates (in practice it outputs $\log\boldsymbol\sigma^2$, which can be any real number).
* **Reparametrisation trick.** A latent vector is drawn from the cloud as $\mathbf z=\boldsymbol\mu+\boldsymbol\sigma\odot\boldsymbol\varepsilon$, with $\boldsymbol\varepsilon\sim\mathcal N(0,I)$ drawn independently of the network. The randomness sits in $\boldsymbol\varepsilon$, so $\mathbf z$ is an ordinary differentiable function of $\boldsymbol\mu$ and $\boldsymbol\sigma$, and backpropagation works through it.
* **Loss.** Two terms per image:
$$\mathcal L(\mathbf x)=\underbrace{\mathrm{BCE}\big(\mathbf x,G(\mathbf z)\big)}_{\text{rebuild the input}}+\underbrace{\tfrac12\sum_{k=1}^{d}\big(\mu_k^2+\sigma_k^2-\ln\sigma_k^2-1\big)}_{\mathrm{KL}(\mathcal N(\boldsymbol\mu,\boldsymbol\sigma^2)\,\|\,\mathcal N(0,I))}.$$
  The binary cross-entropy (BCE, summed over the 784 pixels) suits pixels in $[0,1]$. The KL term pulls every cloud towards $\mathcal N(0,I)$, so the clouds of all images fill one standard Gaussian without gaps. The sum of the two terms is minus a lower bound on $\log p_\theta(\mathbf x)$ (the *evidence lower bound*, ELBO): training the VAE is approximate maximum likelihood.
* **Generation.** Skip the encoder: draw $\mathbf z\sim\mathcal N(0,I)$ and compute $G(\mathbf z)$.

The class below implements this; `cond` adds an optional condition for part 4.

In [10]:
class VAE(nn.Module):
    def __init__(self, D=784, d=2, cond=0, hidden=(512, 256)):
        super().__init__()
        self.d, self.cond = d, cond
        self.enc = mlp([D + cond, *hidden]); self.enc.append(nn.ReLU())
        self.mu, self.logvar = nn.Linear(hidden[-1], d), nn.Linear(hidden[-1], d)
        self.dec = mlp([d + cond, *hidden[::-1], D])                  # outputs logits; sigmoid gives pixels
    def encode(self, x, c=None):
        h = self.enc(x if c is None else torch.cat([x, c], 1))
        return self.mu(h), self.logvar(h)
    def decode(self, z, c=None):
        return self.dec(z if c is None else torch.cat([z, c], 1))
    def forward(self, x, c=None):
        mu, logvar = self.encode(x, c)
        eps = torch.randn_like(mu)                                  # the randomness, outside the network
        z = mu + torch.exp(0.5 * logvar) * eps                      # reparametrisation trick
        return self.decode(z, c), mu, logvar

def vae_loss(logits, x, mu, logvar):
    rec = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / len(x)
    kl = 0.5 * (mu ** 2 + logvar.exp() - logvar - 1).sum() / len(x)
    return rec + kl, rec, kl

def train_vae(model, X, C=None, epochs=20, lr=1e-3, bs=128):
    opt = torch.optim.Adam(model.parameters(), lr)
    hist = []
    for ep in range(epochs):
        perm = torch.randperm(len(X)); tot = np.zeros(3)
        for i in range(0, len(perm), bs):
            j = perm[i:i + bs]; x = X[j]; c = None if C is None else C[j]
            logits, mu, logvar = model(x, c)
            loss, rec, kl = vae_loss(logits, x, mu, logvar)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += np.array([loss.item(), rec.item(), kl.item()]) * len(j)
        hist.append(tot / len(X))
    return np.array(hist)

We train two VAEs on the 60 000 training digits, without labels: one with $d=2$, so that we can draw its latent space, and one with $d=16$, which rebuilds and generates better. The loss is printed per image: reconstruction (BCE) and KL.

In [11]:
rng = seed(1)
t0 = time.time()
vae2 = VAE(d=2); h2 = train_vae(vae2, Xtr_t, epochs=20)
vae16 = VAE(d=16); h16 = train_vae(vae16, Xtr_t, epochs=20)
print(f"trained in {time.time() - t0:.0f} s")
for name, h in [("d = 2", h2), ("d = 16", h16)]:
    print(f"{name}: loss per image {h[-1, 0]:.1f} = reconstruction {h[-1, 1]:.1f} + KL {h[-1, 2]:.1f}")
nums["vae16_loss"], nums["vae16_rec"], nums["vae16_kl"] = [float(v) for v in h16[-1]]

# one test digit and its VAE reconstruction (used in the diagrams of the slides)
vae16.eval()
with torch.no_grad():
    mu, _ = vae16.encode(Xte_t[:1]); x_rec = torch.sigmoid(vae16.decode(mu))[0].numpy()
for arr, name in [(Xte[0], "gen_vae_in.png"), (x_rec, "gen_vae_out.png")]:
    plt.figure(figsize=(2, 2)); plt.imshow(arr.reshape(28, 28), cmap="gray_r"); plt.axis("off")
    plt.savefig(f"{OUT}/{name}", dpi=170, bbox_inches="tight", pad_inches=0.02); plt.show()

trained in 1036 s
d = 2: loss per image 138.9 = reconstruction 132.1 + KL 6.8
d = 16: loss per image 99.9 = reconstruction 80.3 + KL 19.6


/tmp/ipykernel_3869297/3809027835.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{OUT}/{name}", dpi=170, bbox_inches="tight", pad_inches=0.02); plt.show()


### New digits

Left: the decoder of the $d=2$ VAE on a regular grid of $\mathbf z$ over $[-2.5,2.5]^2$, i.e. where $\mathcal N(0,I)$ puts its probability. Every point decodes to a digit, and neighbouring points give similar digits. Right: 32 new digits from the $d=16$ VAE, each from a random $\mathbf z\sim\mathcal N(0,I)$.

In [12]:
vae2.eval()
g = np.linspace(-2.5, 2.5, 15)
zg = torch.tensor([[a, b] for b in g[::-1] for a in g], dtype=torch.float32)
with torch.no_grad(): imgs = torch.sigmoid(vae2.decode(zg)).numpy()
show_grid(imgs, 15, 15, f"{OUT}/gen_vae_grid.png", size=0.5)
torch.manual_seed(10)
with torch.no_grad(): new = torch.sigmoid(vae16.decode(torch.randn(32, 16))).numpy()
show_grid(new, 4, 8, f"{OUT}/gen_vae_samples.png", size=0.9)

# are the new digits copies of training images? distance to the nearest training image
d_new = torch.cdist(torch.tensor(new), Xtr_t).min(1).values.numpy()
d_test = torch.cdist(Xte_t[:32], Xtr_t).min(1).values.numpy()
nums["vae_nn_dist_new"], nums["vae_nn_dist_test"] = float(d_new.mean()), float(d_test.mean())
print(f"mean distance to the nearest training image: generated {d_new.mean():.2f}, real test digits {d_test.mean():.2f}")

/tmp/ipykernel_3869297/2771021278.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(path, dpi=170, bbox_inches="tight"); plt.show()


mean distance to the nearest training image: generated 4.73, real test digits 4.03


The generated digits are not copies: their distance to the nearest training image is comparable to that of real test digits, which the network never saw. The samples are somewhat blurry, a known property of VAEs.

**Try it.** Train the $d=16$ VAE without the KL term (set `kl = 0` in `vae_loss`) and generate from $\mathcal N(0,I)$ again. What happens, and why?

## 5. The conditional VAE

Often we want to say *what* to generate. A conditional VAE gets a known label $\mathbf c$ as an extra input of both networks: $E(\mathbf x,\mathbf c)$ and $G(\mathbf z,\mathbf c)$. The loss is unchanged. Since $\mathbf c$ already tells the decoder which digit to draw, the latent vector $\mathbf z$ only needs to carry what $\mathbf c$ does not say: the **style** (slant, thickness, size).

Here $\mathbf c$ is the one-hot vector of the digit. To generate, we fix $\mathbf c$ and draw $\mathbf z\sim\mathcal N(0,I)$. In the figure, each row is one digit and each column one random $\mathbf z$, the **same** in every row.

In [13]:
rng = seed(2)
Ctr = F.one_hot(torch.tensor(ytr.astype(np.int64)), 10).float()
cvae = VAE(d=8, cond=10); hc = train_vae(cvae, Xtr_t, Ctr, epochs=20); cvae.eval()
print(f"loss per image {hc[-1, 0]:.1f} = reconstruction {hc[-1, 1]:.1f} + KL {hc[-1, 2]:.1f}")
torch.manual_seed(11); zc = torch.randn(8, 8)
rows = []
with torch.no_grad():
    for dgt in range(10):
        c = F.one_hot(torch.full((8,), dgt), 10).float()
        rows.append(torch.sigmoid(cvae.decode(zc, c)).numpy())
show_grid(np.concatenate(rows), 10, 8, f"{OUT}/gen_cvae_mnist.png", size=0.6)

loss per image 97.6 = reconstruction 83.4 + KL 14.2


/tmp/ipykernel_3869297/2771021278.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(path, dpi=170, bbox_inches="tight"); plt.show()


### Testing the generated digits with a classifier

Do the generated digits really show the digit we asked for? We train a small classifier on the **real** MNIST training images (a dense network $784\to256\to10$, cross-entropy, 3 epochs), check its accuracy on the real test images, and then let it label 200 generated images for each requested digit. The fraction labelled as the requested digit measures how well the conditional VAE respects its condition. The same classifier is used again for the GAN in part 7.

In [14]:
rng = seed(4)
clf = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))
optc = torch.optim.Adam(clf.parameters(), 1e-3)
for ep in range(3):
    perm = torch.randperm(len(Xtr_t))
    for i in range(0, len(perm), 128):
        j = perm[i:i + 128]; loss = F.cross_entropy(clf(Xtr_t[j]), torch.tensor(ytr[j.numpy()].astype(np.int64)))
        optc.zero_grad(); loss.backward(); optc.step()
clf.eval()
with torch.no_grad():
    acc = (clf(Xte_t).argmax(1).numpy() == yte).mean()
    agree_cvae = []
    for dgt in range(10):
        cc = F.one_hot(torch.full((200,), dgt), 10).float()
        agree_cvae.append(float((clf(torch.sigmoid(cvae.decode(torch.randn(200, 8), cc))).argmax(1).numpy() == dgt).mean()))
nums["clf_acc"] = float(acc); nums["cvae_agreement"] = [round(a, 3) for a in agree_cvae]
fig, ax = plt.subplots(figsize=(5.5, 2.8))
ax.bar(range(10), agree_cvae, color=blue); ax.set_ylim(0, 1.05); ax.set_xticks(range(10))
ax.set_xlabel("requested digit"); ax.set_ylabel("recognised as requested"); clean(ax)
plt.savefig(f"{OUT}/gen_cvae_mnist_check.png", dpi=170, bbox_inches="tight"); plt.show()
print(f"classifier accuracy on real test digits: {acc:.3f}")
print("fraction of generated digits recognised as the requested digit:", np.round(agree_cvae, 2), " mean", round(float(np.mean(agree_cvae)), 3))

classifier accuracy on real test digits: 0.971
fraction of generated digits recognised as the requested digit: [0.98 0.94 0.98 0.98 0.96 0.94 0.96 0.96 0.96 0.96]  mean 0.963


/tmp/ipykernel_3869297/2990466343.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{OUT}/gen_cvae_mnist_check.png", dpi=170, bbox_inches="tight"); plt.show()


**Try it.** Keep $\mathbf c$ fixed and move along one latent coordinate, $z_1$ from $-3$ to $3$, with the others at 0. Which property of the digit changes?

## 6. Generating blood-cell images of a chosen type

The same conditional VAE idea on BloodMNIST (Lecture 4): $28\times28$ colour images of eight types of normal blood cells. The condition $\mathbf c$ is the one-hot cell type. Images have 3 colour channels, so encoder and decoder are convolutional, like the CNNs of Lecture 4.

Do the generated cells really look like their type? We check it with a small CNN classifier trained on the real images: we generate 100 cells of each type and count how many the classifier assigns to the type we asked for.

In [15]:
import urllib.request
bpath = "data/bloodmnist.npz"
if not os.path.exists(bpath):
    bpath = "data/bloodmnist.npz"
    if not os.path.exists(bpath):
        os.makedirs("data", exist_ok=True)
        urllib.request.urlretrieve("https://zenodo.org/records/10519652/files/bloodmnist.npz?download=1", bpath)
bd = np.load(bpath)
names = ["basophil", "eosinophil", "erythroblast", "immature granulocyte", "lymphocyte", "monocyte", "neutrophil", "platelet"]
to_t = lambda a: torch.tensor(a.astype(np.float32) / 255).permute(0, 3, 1, 2)      # (n, 3, 28, 28)
Btr, Bte = to_t(bd["train_images"]), to_t(bd["test_images"])
bytr, byte = bd["train_labels"].ravel(), bd["test_labels"].ravel()
Cb = F.one_hot(torch.tensor(bytr.astype(np.int64)), 8).float()
print("training images:", tuple(Btr.shape), " test images:", tuple(Bte.shape))

class ConvCVAE(nn.Module):
    def __init__(self, d=16, cond=8):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(3 + cond, 32, 3, 2, 1), nn.ReLU(), nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(), nn.Flatten())
        self.mu, self.logvar = nn.Linear(64 * 7 * 7, d), nn.Linear(64 * 7 * 7, d)
        self.fc = nn.Linear(d + cond, 64 * 7 * 7)
        self.dec = nn.Sequential(nn.ReLU(), nn.Unflatten(1, (64, 7, 7)), nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1), nn.ReLU(),
                                 nn.ConvTranspose2d(32, 3, 3, 2, 1, output_padding=1))
        self.cond = cond
    def encode(self, x, c):
        cmap_ = c[:, :, None, None].expand(-1, -1, x.shape[2], x.shape[3])        # the condition as extra input channels
        h = self.enc(torch.cat([x, cmap_], 1)); return self.mu(h), self.logvar(h)
    def decode(self, z, c): return self.dec(self.fc(torch.cat([z, c], 1)))       # logits of the pixels
    def forward(self, x, c):
        mu, logvar = self.encode(x, c); z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decode(z, c), mu, logvar

rng = seed(7); t0 = time.time()
bcvae = ConvCVAE(); optb = torch.optim.Adam(bcvae.parameters(), 1e-3)
for ep in range(30):
    perm = torch.randperm(len(Btr))
    for i in range(0, len(perm), 128):
        j = perm[i:i + 128]; logits, mu, logvar = bcvae(Btr[j], Cb[j])
        loss, rec, kl = vae_loss(logits.flatten(1), Btr[j].flatten(1), mu, logvar)
        optb.zero_grad(); loss.backward(); optb.step()
bcvae.eval(); print(f"trained in {time.time() - t0:.0f} s; last batch: reconstruction {rec.item():.1f} + KL {kl.item():.1f}")

# a classifier of the real cells, for the check
rng = seed(8)
clfb = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                     nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 8))
optk = torch.optim.Adam(clfb.parameters(), 1e-3); ybt = torch.tensor(bytr.astype(np.int64))
for ep in range(8):
    perm = torch.randperm(len(Btr))
    for i in range(0, len(perm), 128):
        j = perm[i:i + 128]; loss = F.cross_entropy(clfb(Btr[j]), ybt[j]); optk.zero_grad(); loss.backward(); optk.step()
clfb.eval()
with torch.no_grad(): accb = (clfb(Bte).argmax(1).numpy() == byte).mean()
nums["blood_clf_test_acc"] = float(accb)

torch.manual_seed(13)
with torch.no_grad():
    zb = torch.randn(8, 16); rows = []
    for c in range(8):
        rows.append(torch.sigmoid(bcvae.decode(zb, F.one_hot(torch.full((8,), c), 8).float())).permute(0, 2, 3, 1).numpy())
    agree = []
    for c in range(8):
        g = torch.sigmoid(bcvae.decode(torch.randn(100, 16), F.one_hot(torch.full((100,), c), 8).float()))
        agree.append(float((clfb(g).argmax(1).numpy() == c).mean()))
nums["blood_cvae_class_agreement"] = [round(a, 2) for a in agree]
fig, axs = plt.subplots(8, 9, figsize=(8.2, 7.6), gridspec_kw=dict(width_ratios=[1] * 8 + [1.6]))
for c in range(8):
    for j in range(8):
        axs[c, j].imshow(rows[c][j]); axs[c, j].axis("off")
    axs[c, 8].axis("off"); axs[c, 8].text(0.05, 0.5, f"{names[c]}\n{100 * agree[c]:.0f}% recognised", va="center", fontsize=8)
plt.subplots_adjust(wspace=0.04, hspace=0.06); plt.savefig(f"{OUT}/gen_cvae_blood.png", dpi=170, bbox_inches="tight"); plt.show()
print(f"classifier test accuracy {accb:.3f}"); print({n: a for n, a in zip(names, agree)})

training images: (11959, 3, 28, 28)  test images: (3421, 3, 28, 28)


trained in 505 s; last batch: reconstruction 1161.0 + KL 14.5


classifier test accuracy 0.838
{'basophil': 0.04, 'eosinophil': 0.81, 'erythroblast': 0.98, 'immature granulocyte': 0.86, 'lymphocyte': 0.14, 'monocyte': 0.26, 'neutrophil': 0.14, 'platelet': 1.0}


/tmp/ipykernel_3869297/6733656.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.subplots_adjust(wspace=0.04, hspace=0.06); plt.savefig(f"{OUT}/gen_cvae_blood.png", dpi=170, bbox_inches="tight"); plt.show()


Each row is one requested cell type, each column one random $\mathbf z$ (the same in every row). The percentage is the fraction of 100 generated cells of that type that the classifier, trained on real cells, assigns to the requested type.

**Try it.** Generate cells of one type with several values of a single latent coordinate. What changes in the image?

## 7. Generative adversarial networks

A GAN learns to sample without ever writing $p(\mathbf x)$ down. Two networks:

* the **generator** $G$ turns noise $\mathbf z\sim\mathcal N(0,I)$ into a sample $\tilde{\mathbf x}=G(\mathbf z)$; it never sees the data;
* the **discriminator** $D$ is a classifier: $D(\mathbf x)$ is the probability that $\mathbf x$ is real, not generated.

$D$ is trained with the binary cross-entropy to label real images 1 and generated images 0; $G$ is trained to make $D$ wrong, and in practice it minimises $-\log D(G(\mathbf z))$ (the *non-saturating* loss, which gives $G$ useful gradients even when $D$ wins easily). The two steps alternate. When $D$ is as good as possible, improving $G$ moves the distribution of the generated samples towards that of the data.

GANs are hard to train: there is no single loss that decreases, the two networks can oscillate, and the generator can collapse onto a few kinds of samples (*mode collapse*). We train a small GAN and look at all of this.

The details matter. We use the common recipe for dense GANs: LeakyReLU activations in both networks, a $\tanh$ output of the generator with the pixels rescaled to $[-1,1]$, and *dropout* in the discriminator, which keeps it from winning too early. Without them (ReLU, sigmoid output, no dropout) the discriminator wins within the first epoch, its loss drops to about 0.01, and the generator collapses onto one blob (see **Try it**).

In [16]:
rng = seed(3)
def gen(n_in):                                                  # generator: noise (+ condition) -> 784 pixels in [-1, 1]
    return nn.Sequential(nn.Linear(n_in, 256), nn.LeakyReLU(0.2), nn.Linear(256, 512), nn.LeakyReLU(0.2),
                         nn.Linear(512, 1024), nn.LeakyReLU(0.2), nn.Linear(1024, 784), nn.Tanh())
def disc(n_in):                                                 # discriminator: image (+ condition) -> one logit
    return nn.Sequential(nn.Linear(n_in, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
                         nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3), nn.Linear(256, 1))
to01 = lambda x: (x + 1) / 2                                    # back to pixels in [0, 1], for plots and the classifier
G, D = gen(64), disc(784)
optG = torch.optim.Adam(G.parameters(), 2e-4, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), 2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()
z_fixed = torch.randn(16, 64)
snap, lossD_hist, lossG_hist = {}, [], []
EPOCHS = 30; t0 = time.time()
for ep in range(1, EPOCHS + 1):
    perm = torch.randperm(len(Xtr_t)); lD = lG = 0.0
    for i in range(0, len(perm), 128):
        real = Xtr_t[perm[i:i + 128]] * 2 - 1; n = len(real)
        fake = G(torch.randn(n, 64))
        # discriminator step: real -> 1, generated -> 0
        lossD = bce(D(real), torch.ones(n, 1)) + bce(D(fake.detach()), torch.zeros(n, 1))
        optD.zero_grad(); lossD.backward(); optD.step()
        # generator step: make D call the generated images real
        lossG = bce(D(fake), torch.ones(n, 1))
        optG.zero_grad(); lossG.backward(); optG.step()
        lD += lossD.item() * n; lG += lossG.item() * n
    lossD_hist.append(lD / len(perm)); lossG_hist.append(lG / len(perm))
    if ep in (1, 5, EPOCHS):
        G.eval()
        with torch.no_grad(): snap[ep] = to01(G(z_fixed)).numpy()
        G.train()
print(f"{EPOCHS} epochs in {time.time() - t0:.0f} s")

fig, axs = plt.subplots(3, 16, figsize=(12, 2.6))
for r, ep in enumerate(snap):
    for j in range(16):
        axs[r, j].imshow(snap[ep][j].reshape(28, 28), cmap="gray_r", vmin=0, vmax=1); axs[r, j].axis("off")
    axs[r, 0].text(-6, 14, f"epoch {ep}", ha="right", va="center", fontsize=9)
plt.subplots_adjust(wspace=0.03, hspace=0.05); plt.savefig(f"{OUT}/gen_gan_progress.png", dpi=170, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(range(1, EPOCHS + 1), lossD_hist, "o-", ms=3, color=blue, label="discriminator")
ax.plot(range(1, EPOCHS + 1), lossG_hist, "s-", ms=3, color=orange, label="generator")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(frameon=False); clean(ax)
plt.savefig(f"{OUT}/gen_gan_losses.png", dpi=170, bbox_inches="tight"); plt.show()

30 epochs in 933 s


/tmp/ipykernel_3869297/66087476.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.subplots_adjust(wspace=0.03, hspace=0.05); plt.savefig(f"{OUT}/gen_gan_progress.png", dpi=170, bbox_inches="tight"); plt.show()


/tmp/ipykernel_3869297/66087476.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{OUT}/gen_gan_losses.png", dpi=170, bbox_inches="tight"); plt.show()


The samples improve from epoch to epoch, but the loss curves do not show it: the two losses measure a game, not the quality of the samples.

### Mode Collapse

The MNIST classifier of part 5 labels 2000 new samples of the GAN and 2000 of the VAE. A generator that covers the data produces every digit about 10% of the time.

In [17]:
with torch.no_grad():
    G.eval(); gan_lab = clf(to01(G(torch.randn(2000, 64)))).argmax(1).numpy()
    vae_lab = clf(torch.sigmoid(vae16.decode(torch.randn(2000, 16)))).argmax(1).numpy()
fg, fv = np.bincount(gan_lab, minlength=10) / 2000, np.bincount(vae_lab, minlength=10) / 2000
nums["gan_digit_frac"], nums["vae_digit_frac"] = fg.round(3).tolist(), fv.round(3).tolist()
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(np.arange(10) - 0.2, fg, 0.4, color=orange, label="GAN"); ax.bar(np.arange(10) + 0.2, fv, 0.4, color=blue, label="VAE")
ax.axhline(0.1, color="k", ls=":", lw=1); ax.set_xticks(range(10)); ax.set_xlabel("digit assigned by the classifier"); ax.set_ylabel("fraction of samples")
ax.legend(frameon=False); clean(ax)
plt.savefig(f"{OUT}/gen_gan_modes.png", dpi=170, bbox_inches="tight"); plt.show()
print("GAN:", fg.round(3)); print("VAE:", fv.round(3))

GAN: [0.108 0.152 0.1   0.099 0.086 0.098 0.092 0.103 0.064 0.098]
VAE: [0.094 0.082 0.129 0.118 0.092 0.084 0.074 0.098 0.116 0.112]


/tmp/ipykernel_3869297/3982532920.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{OUT}/gen_gan_modes.png", dpi=170, bbox_inches="tight"); plt.show()


### The Best Discriminator and the Wasserstein Distance

For a fixed generator with density $p_G$, the discriminator that maximises $V(D,G)$ is known in closed form (Goodfellow et al. 2014, Proposition 1):
$$D^*(\mathbf x)=\frac{p_{\text{data}}(\mathbf x)}{p_{\text{data}}(\mathbf x)+p_G(\mathbf x)} .$$
It is the probability that a point at $\mathbf x$ is real when real and generated points are equally frequent. Where the generator puts too little probability, $D^*>1/2$; where it puts too much, $D^*<1/2$; when $p_G=p_{\text{data}}$, $D^*=1/2$ everywhere and the discriminator can only guess.

The **Wasserstein-1 distance** between two distributions on a line is the area between their cumulative distribution functions, $W(p,q)=\int|F_p(x)-F_q(x)|\,dx$: the probability mass to be moved times the distance it travels. Two narrow bumps, $p$ at 0 and $q$ at $\theta$: the Jensen–Shannon divergence (what the original GAN loss measures at the best $D$) is stuck at $\log2$ as soon as they stop overlapping, whatever $\theta$; the Wasserstein distance is $\theta$ and still tells how far apart they are.

In [18]:
from scipy.stats import norm
xd = np.linspace(-6, 6, 2401); dxd = xd[1] - xd[0]
pdat = 0.5 * norm.pdf(xd, -2, 0.6) + 0.5 * norm.pdf(xd, 1.5, 1.0)
pG = norm.pdf(xd, 0.5, 1.2)
fig, axs = plt.subplots(2, 2, figsize=(10, 4.6), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]})
for k, (q, ttl) in enumerate([(pG, r"generator not yet good: $p_G \neq p_{\mathrm{data}}$"), (pdat, r"generator perfect: $p_G = p_{\mathrm{data}}$")]):
    a, b = axs[0, k], axs[1, k]
    a.plot(xd, pdat, color=blue, lw=2.2, label=r"$p_{\mathrm{data}}$"); a.plot(xd, q, color=orange, lw=2, ls="--", label="$p_G$")
    a.set_title(ttl, fontsize=11); a.legend(frameon=False, fontsize=10, loc="upper right")
    b.plot(xd, pdat / (pdat + q + 1e-300), color=navy, lw=2.2); b.axhline(0.5, color="grey", ls=":", lw=1)
    b.set_ylim(0, 1.05); b.set_xlim(-5, 5); b.set_xlabel("$x$"); b.set_ylabel(r"$D^*(x)$")
    for c_ in (a, b): clean(c_)
plt.tight_layout(); plt.savefig(f"{OUT}/gen_gan_optimal_D.png", dpi=170, bbox_inches="tight"); plt.show()

def kld(p, q): return float((p * (np.log(p + 1e-300) - np.log(q + 1e-300))).sum() * dxd)
w = 0.1; thetas = np.linspace(0, 3, 121); JS, W = [], []
for th_ in thetas:
    p_, q_ = norm.pdf(xd, 0, w), norm.pdf(xd, th_, w); m_ = 0.5 * (p_ + q_)
    JS.append(0.5 * kld(p_, m_) + 0.5 * kld(q_, m_))
    W.append(float(np.abs(np.cumsum(p_) - np.cumsum(q_)).sum() * dxd * dxd))
nums["js_theta1"], nums["js_theta2"] = JS[40], JS[80]; nums["w_theta1"], nums["w_theta2"] = W[40], W[80]
fig, axs = plt.subplots(1, 2, figsize=(10, 3.0))
axs[0].plot(xd, norm.pdf(xd, 0, w), color=blue, lw=2, label="$p$"); axs[0].plot(xd, norm.pdf(xd, 2, w), color=orange, lw=2, label=r"$q$, $\theta = 2$")
axs[0].annotate("", xy=(1.85, 2.0), xytext=(0.15, 2.0), arrowprops=dict(arrowstyle="->", color="k", lw=1.5))
axs[0].set_xlim(-1, 3); axs[0].set_xlabel("$x$"); axs[0].legend(frameon=False, fontsize=9); axs[0].set_title("two non-overlapping distributions", fontsize=11)
axs[1].plot(thetas, JS, color=navy, lw=2.2, label="Jensen–Shannon"); axs[1].plot(thetas, W, color=orange, lw=2.2, label="Wasserstein-1")
axs[1].axhline(np.log(2), color="grey", ls=":", lw=1); axs[1].text(2.2, np.log(2) + 0.08, r"$\log 2$", fontsize=10)
axs[1].set_xlabel(r"shift $\theta$"); axs[1].legend(frameon=False, fontsize=9); axs[1].set_title("distance between $p$ and $q$", fontsize=11)
for a in axs: clean(a)
plt.tight_layout(); plt.savefig(f"{OUT}/gen_wasserstein_js.png", dpi=170, bbox_inches="tight"); plt.show()
print(f"theta = 1: JS = {JS[40]:.3f}, W = {W[40]:.3f};  theta = 2: JS = {JS[80]:.3f}, W = {W[80]:.3f};  log 2 = {np.log(2):.3f}")

/tmp/ipykernel_3869297/3206031842.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_gan_optimal_D.png", dpi=170, bbox_inches="tight"); plt.show()


theta = 1: JS = 0.693, W = 1.000;  theta = 2: JS = 0.693, W = 2.000;  log 2 = 0.693


/tmp/ipykernel_3869297/3206031842.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_wasserstein_js.png", dpi=170, bbox_inches="tight"); plt.show()


Once the bumps no longer overlap, the Jensen–Shannon divergence is constant: its gradient with respect to $\theta$ is zero, and a generator trained on it learns nothing about where to move. The Wasserstein distance grows linearly with $\theta$ and always points the way.

### A conditional GAN

As for the VAE, a condition $\mathbf c$ (here the one-hot digit) is given to **both** networks: the generator $G(\mathbf z,\mathbf c)$ draws an image of the requested digit, and the discriminator $D(\mathbf x,\mathbf c)$ judges the *pair*: is this a real image of digit $\mathbf c$? A realistic 3 sent with the condition "7" is therefore rejected, and $G$ must respect the condition. We check the result with the same MNIST classifier.

In [19]:
rng = seed(9)
Gc, Dc = gen(64 + 10), disc(784 + 10)
oG = torch.optim.Adam(Gc.parameters(), 2e-4, betas=(0.5, 0.999)); oD = torch.optim.Adam(Dc.parameters(), 2e-4, betas=(0.5, 0.999))
ytr_t = torch.tensor(ytr.astype(np.int64)); t0 = time.time()
for ep in range(30):
    perm = torch.randperm(len(Xtr_t))
    for i in range(0, len(perm), 128):
        j = perm[i:i + 128]; real = Xtr_t[j] * 2 - 1; c = F.one_hot(ytr_t[j], 10).float(); n = len(j)
        fake = Gc(torch.cat([torch.randn(n, 64), c], 1))
        lossD = bce(Dc(torch.cat([real, c], 1)), torch.ones(n, 1)) + bce(Dc(torch.cat([fake.detach(), c], 1)), torch.zeros(n, 1))
        oD.zero_grad(); lossD.backward(); oD.step()
        lossG = bce(Dc(torch.cat([fake, c], 1)), torch.ones(n, 1))
        oG.zero_grad(); lossG.backward(); oG.step()
print(f"trained in {time.time() - t0:.0f} s")
torch.manual_seed(14); zc = torch.randn(8, 64); rows, agree = [], []; Gc.eval()
with torch.no_grad():
    for dgt in range(10):
        c = F.one_hot(torch.full((8,), dgt), 10).float(); rows.append(to01(Gc(torch.cat([zc, c], 1))).numpy())
        cc = F.one_hot(torch.full((200,), dgt), 10).float()
        agree.append(float((clf(to01(Gc(torch.cat([torch.randn(200, 64), cc], 1)))).argmax(1).numpy() == dgt).mean()))
nums["cgan_agreement"] = [round(a, 3) for a in agree]
show_grid(np.concatenate(rows), 10, 8, f"{OUT}/gen_cgan_mnist.png", size=0.6)
print("fraction recognised as the requested digit, conditional GAN:", np.round(agree, 2), " mean", round(float(np.mean(agree)), 3))
print("conditional VAE (part 5), for comparison:", np.round(agree_cvae, 2), " mean", round(float(np.mean(agree_cvae)), 3))

trained in 991 s


/tmp/ipykernel_3869297/2771021278.py:35: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axs = plt.subplots(nrow, ncol, figsize=(ncol * size, nrow * size))


fraction recognised as the requested digit, conditional GAN: [1.   0.98 0.98 0.92 0.98 0.92 0.94 1.   0.93 0.96]  mean 0.961
conditional VAE (part 5), for comparison: [0.98 0.94 0.98 0.98 0.96 0.94 0.96 0.96 0.96 0.96]  mean 0.963


/tmp/ipykernel_3869297/2771021278.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(path, dpi=170, bbox_inches="tight"); plt.show()


**Try it.** Replace the generator by `nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 512), nn.ReLU(), nn.Linear(512, 784), nn.Sigmoid())`, remove the dropout and the rescaling to $[-1,1]$, and watch the discriminator loss and the samples after one epoch.

**Try it.** Train the GAN with the *saturating* generator loss, i.e. minimise $\log(1-D(G(\mathbf z)))$ instead of $-\log D(G(\mathbf z))$. Does training still start?

## 8. Diffusion models

A diffusion model learns to **undo noise**.

* **Forward process (no learning).** Starting from a data point $\mathbf x_0$, add a little Gaussian noise at each of $T$ steps; after many steps nothing is left but noise. With a fixed schedule $\beta_1,\dots,\beta_T$ the noisy point at step $t$ can be written directly, $\mathbf x_t=\sqrt{\bar\alpha_t}\,\mathbf x_0+\sqrt{1-\bar\alpha_t}\,\boldsymbol\varepsilon$, with $\bar\alpha_t=\prod_{s\le t}(1-\beta_s)$ and $\boldsymbol\varepsilon\sim\mathcal N(0,I)$. The schedule must make $\bar\alpha_T\approx0$: generation starts from pure noise, so the forward process has to end there too. (A schedule that leaves part of the data at step $T$ makes the network start from a state it never saw in training, and the samples come out as scrawls.)
* **Training.** A network $\boldsymbol\varepsilon_\theta(\mathbf x_t,t)$ learns to predict the noise that was added: minimise $|\boldsymbol\varepsilon-\boldsymbol\varepsilon_\theta(\mathbf x_t,t)|^2$ over random data points, steps and noises. An ordinary regression.
* **Generation.** Start from pure noise $\mathbf x_T\sim\mathcal N(0,I)$ and remove the predicted noise one step at a time down to $\mathbf x_0$.

We use 2D data (two half-moons), where every step can be drawn.

In [20]:
from sklearn.datasets import make_moons
rng = seed(5)
Xm = torch.tensor(make_moons(4000, noise=0.05, random_state=0)[0], dtype=torch.float32)
Xm = (Xm - Xm.mean(0)) / Xm.std(0)
T = 100
beta = torch.linspace(1e-4, 0.1, T); alpha = 1 - beta; abar = torch.cumprod(alpha, 0)
print(f"after T = {T} steps the data are scaled by sqrt(abar_T) = {abar[-1].sqrt():.3f}: almost nothing is left but noise")
class EpsNet(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 + 16, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t):
        f = torch.arange(8) + 1.0
        emb = torch.cat([torch.sin(t[:, None] / T * f * np.pi), torch.cos(t[:, None] / T * f * np.pi)], 1)  # step t as 16 numbers
        return self.net(torch.cat([x, emb], 1))
net = EpsNet(); opt = torch.optim.Adam(net.parameters(), 2e-3)
t0 = time.time(); losses = []
for it in range(6000):
    x0 = Xm[torch.randint(0, len(Xm), (256,))]
    t = torch.randint(0, T, (256,)); eps = torch.randn_like(x0)
    xt = abar[t].sqrt()[:, None] * x0 + (1 - abar[t]).sqrt()[:, None] * eps
    loss = ((eps - net(xt, t.float())) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
print(f"trained in {time.time() - t0:.0f} s; final loss {np.mean(losses[-200:]):.3f}")

# forward process of the data, and the learned reverse process from fresh noise
show_t = [0, 20, 50, 99]
with torch.no_grad():
    eps = torch.randn_like(Xm[:1500])
    fwd = {t: (abar[t].sqrt() * Xm[:1500] + (1 - abar[t]).sqrt() * eps).numpy() for t in show_t}
    x = torch.randn(1500, 2); rev = {99: x.numpy().copy()}
    for t in range(T - 1, -1, -1):
        e = net(x, torch.full((len(x),), float(t)))
        x = (x - beta[t] / (1 - abar[t]).sqrt() * e) / alpha[t].sqrt()
        if t > 0: x = x + beta[t].sqrt() * torch.randn_like(x)
        if t in (50, 20, 0): rev[t] = x.numpy().copy()
fig, axs = plt.subplots(2, 4, figsize=(11, 5.4))
for k, t in enumerate(show_t):
    axs[0, k].scatter(*fwd[t].T, s=1, color=blue); axs[0, k].set_title(f"forward, step {t + 1 if t else 0}", fontsize=10)
for k, t in enumerate([99, 50, 20, 0]):
    axs[1, k].scatter(*rev[t].T, s=1, color=orange); axs[1, k].set_title(["pure noise", "reverse, step 50", "reverse, step 20", "generated"][k], fontsize=10)
for a in axs.ravel(): a.set_xlim(-3, 3); a.set_ylim(-3, 3); a.set_xticks([]); a.set_yticks([]); a.set_aspect("equal")
plt.tight_layout(); plt.savefig(f"{OUT}/gen_diffusion_forward_reverse.png", dpi=170, bbox_inches="tight"); plt.show()

after T = 100 steps the data are scaled by sqrt(abar_T) = 0.075: almost nothing is left but noise


trained in 24 s; final loss 0.322


/tmp/ipykernel_3869297/3678782789.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_diffusion_forward_reverse.png", dpi=170, bbox_inches="tight"); plt.show()


### The Forward Process and the Score

One step of the forward process shrinks the point a little and adds a little noise,
$$\mathbf x_t=\sqrt{1-\beta_t}\,\mathbf x_{t-1}+\sqrt{\beta_t}\,\boldsymbol\varepsilon_t,\qquad\boldsymbol\varepsilon_t\sim\mathcal N(0,I).$$
The two factors are chosen so that their squares add up to 1: data with variance 1 keep variance 1 at every step. Combining $t$ steps (a sum of independent Gaussians is Gaussian) gives $\mathbf x_t=\sqrt{\bar\alpha_t}\,\mathbf x_0+\sqrt{1-\bar\alpha_t}\,\boldsymbol\varepsilon$ with $\bar\alpha_t=\prod_{s\le t}(1-\beta_s)$.

In 1D we can follow the whole distribution: data at $\pm1$ (two narrow peaks) turn into one broad Gaussian.

The trained noise predictor has a geometric meaning. For Gaussian noising, the best possible prediction of the noise satisfies $\boldsymbol\varepsilon_\theta(\mathbf x_t,t)\approx-\sqrt{1-\bar\alpha_t}\,\nabla_{\mathbf x}\log p_t(\mathbf x_t)$, where $p_t$ is the distribution of the noisy data at step $t$. So $-\boldsymbol\varepsilon_\theta/\sqrt{1-\bar\alpha_t}$ is the **score** $\nabla\log p_t$: an arrow that points towards higher data density. We draw these arrows for the two-moons network trained above.

In [21]:
from scipy.stats import norm
sq = abar.sqrt().numpy(); xx1 = np.linspace(-3.5, 3.5, 701); s0 = 0.1
def p_t(t):                                                    # density of x_t for data 0.5 N(-1, s0^2) + 0.5 N(1, s0^2)
    m_, v_ = sq[t], sq[t] ** 2 * s0 ** 2 + (1 - sq[t] ** 2)
    return 0.5 * norm.pdf(xx1, -m_, np.sqrt(v_)) + 0.5 * norm.pdf(xx1, m_, np.sqrt(v_))
nums["moons_sqrt_abar_T"] = float(sq[-1])
nums["ddpm_sqrt_abar_T"] = float(torch.cumprod(1 - torch.linspace(5e-4, 0.1, 200), 0)[-1].sqrt())
fig, axs = plt.subplots(1, 2, figsize=(11, 3.3), gridspec_kw={"width_ratios": [1.25, 1]})
dens = np.stack([p_t(t) for t in range(T)], 1)
axs[0].imshow(dens, origin="lower", extent=[1, T, xx1[0], xx1[-1]], aspect="auto", cmap="Blues", vmax=1.0)
axs[0].set_xlabel("step $t$"); axs[0].set_ylabel("$x_t$"); axs[0].set_title("density of $x_t$ along the forward process", fontsize=11)
for t, col in zip([0, 10, 30, 99], [navy, blue, orange, "grey"]):
    axs[1].plot(xx1, p_t(t), color=col, lw=2, label=rf"$t = {t + 1}$: $\sqrt{{\bar\alpha_t}} = {sq[t]:.2f}$")
axs[1].plot(xx1, norm.pdf(xx1), color="k", lw=1, ls=":", label=r"$\mathcal{N}(0, 1)$")
axs[1].set_ylim(0, 1.6); axs[1].set_xlabel("$x$"); axs[1].legend(frameon=False, fontsize=8.5); clean(axs[1])
axs[1].set_title("$p_t(x)$ at four steps", fontsize=11)
plt.tight_layout(); plt.savefig(f"{OUT}/gen_diffusion_1d.png", dpi=170, bbox_inches="tight"); plt.show()

g1 = np.linspace(-2.6, 2.6, 17); GX, GY = np.meshgrid(g1, g1); grid_t = torch.tensor(np.c_[GX.ravel(), GY.ravel()], dtype=torch.float32)
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
torch.manual_seed(23); e0 = torch.randn_like(Xm[:1500])
for a, t in zip(axs, [5, 25, 70]):
    xt_ = (abar[t].sqrt() * Xm[:1500] + (1 - abar[t]).sqrt() * e0).numpy()
    with torch.no_grad(): sc = (-net(grid_t, torch.full((len(grid_t),), float(t))) / (1 - abar[t]).sqrt()).numpy()
    a.scatter(*xt_.T, s=1, color=blue, alpha=0.5)
    a.quiver(GX, GY, sc[:, 0], sc[:, 1], color=orange, angles="xy", width=0.004)
    a.set_title(rf"step $t = {t + 1}$, $\sqrt{{\bar\alpha_t}} = {abar[t].sqrt():.2f}$", fontsize=11)
    a.set_xlim(-2.8, 2.8); a.set_ylim(-2.8, 2.8); a.set_xticks([]); a.set_yticks([]); a.set_aspect("equal")
plt.tight_layout(); plt.savefig(f"{OUT}/gen_diffusion_score.png", dpi=170, bbox_inches="tight"); plt.show()
print(f"sqrt(abar_T): two moons (T = 100) {nums['moons_sqrt_abar_T']:.4f};  MNIST (T = 200) {nums['ddpm_sqrt_abar_T']:.5f}")

/tmp/ipykernel_3869297/1496325925.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_diffusion_1d.png", dpi=170, bbox_inches="tight"); plt.show()


sqrt(abar_T): two moons (T = 100) 0.0750;  MNIST (T = 200) 0.00551


/tmp/ipykernel_3869297/1496325925.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_diffusion_score.png", dpi=170, bbox_inches="tight"); plt.show()


Blue: noisy data at three steps. Orange: the learned arrows $-\boldsymbol\varepsilon_\theta/\sqrt{1-\bar\alpha_t}$. At every step the arrows point from empty regions towards the noisy data, which change from two moons (small $t$) to one broad cloud around the origin (large $t$). Generation follows these arrows from step $T$ down to 1, with a little fresh noise at each step.

### Diffusion on MNIST

The same algorithm on images. The noise-prediction network is now a small **U-Net**: convolutions that shrink the image ($28\to14\to7$ pixels) and grow it back, with *skip connections* that pass the fine details from the shrinking to the growing path; the step $t$ enters as an extra input, added to the features. $T=200$ noise steps, trained on the 60 000 training digits to predict the added noise.

Training (15 epochs) takes about half an hour on a CPU. For generation we use an *exponential moving average* of the weights over the training steps, a standard trick that smooths out the noise of the last updates; the trained weights are saved to `results/ddpm_mnist.pt` and loaded on later runs. Generating means 200 passes of the network, from pure noise to a digit. We check the generated digits with the MNIST classifier of part 5.

In [22]:
class TimeEmb(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim; self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))
    def forward(self, t):                                         # t: (n,) step numbers -> (n, dim)
        f = torch.exp(-np.log(1000) * torch.arange(self.dim // 2) / (self.dim // 2))
        a = t[:, None].float() * f[None]
        return self.mlp(torch.cat([a.sin(), a.cos()], 1))
class Block(nn.Module):
    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.c1, self.c2 = nn.Conv2d(cin, cout, 3, padding=1), nn.Conv2d(cout, cout, 3, padding=1)
        self.t = nn.Linear(tdim, cout); self.n1, self.n2 = nn.GroupNorm(8, cout), nn.GroupNorm(8, cout)
    def forward(self, x, te):
        h = F.silu(self.n1(self.c1(x))) + self.t(te)[:, :, None, None]   # the step t enters here
        return F.silu(self.n2(self.c2(h)))
class UNet(nn.Module):
    def __init__(self, ch=32, tdim=64):
        super().__init__()
        self.te = TimeEmb(tdim)
        self.d1, self.d2, self.mid = Block(1, ch, tdim), Block(ch, 2 * ch, tdim), Block(2 * ch, 2 * ch, tdim)
        self.u2, self.u1 = Block(4 * ch, ch, tdim), Block(2 * ch, ch, tdim)
        self.out = nn.Conv2d(ch, 1, 1)
    def forward(self, x, t):
        te = self.te(t)
        h1 = self.d1(x, te)                                       # 28 x 28
        h2 = self.d2(F.avg_pool2d(h1, 2), te)                     # 14 x 14
        m = self.mid(F.avg_pool2d(h2, 2), te)                     # 7 x 7
        u2 = self.u2(torch.cat([F.interpolate(m, scale_factor=2), h2], 1), te)    # skip connection
        u1 = self.u1(torch.cat([F.interpolate(u2, scale_factor=2), h1], 1), te)   # skip connection
        return self.out(u1)

TD = 200
bD = torch.linspace(5e-4, 0.1, TD); aD = 1 - bD; abD = torch.cumprod(aD, 0)        # DDPM schedule (1e-4 ... 0.02 for 1000 steps), rescaled to 200 steps
print(f"sqrt(abar_T) = {abD[-1].sqrt():.5f}: the last noisy image contains no trace of the digit")
Ximg = Xtr_t.reshape(-1, 1, 28, 28) * 2 - 1                      # pixels in [-1, 1]
unet = UNet(); wpath = f"{RES}/ddpm_mnist.pt"
if os.path.exists(wpath):
    unet.load_state_dict(torch.load(wpath)); print("loaded trained weights from", wpath)
else:
    import copy
    rng = seed(15); optu = torch.optim.Adam(unet.parameters(), 1e-3); t0 = time.time()
    ema = copy.deepcopy(unet)                                     # moving average of the weights, used for generation
    for ep in range(15):
        perm = torch.randperm(len(Ximg)); tot = 0.0
        for i in range(0, len(perm), 128):
            x0 = Ximg[perm[i:i + 128]]; t = torch.randint(0, TD, (len(x0),)); eps = torch.randn_like(x0)
            xt = abD[t].sqrt()[:, None, None, None] * x0 + (1 - abD[t]).sqrt()[:, None, None, None] * eps
            loss = ((eps - unet(xt, t)) ** 2).mean(); optu.zero_grad(); loss.backward(); optu.step(); tot += loss.item() * len(x0)
            with torch.no_grad():
                for pe, pu in zip(ema.parameters(), unet.parameters()): pe.mul_(0.999).add_(pu, alpha=0.001)
        print(f"epoch {ep + 1}: noise-prediction loss {tot / len(Ximg):.4f}   ({time.time() - t0:.0f} s)")
    nums["ddpm_train_seconds"] = float(time.time() - t0)
    unet = ema; torch.save(unet.state_dict(), wpath)
unet.eval()

torch.manual_seed(16); t0 = time.time()
with torch.no_grad():
    x = torch.randn(64, 1, 28, 28); traj = {TD: x[0, 0].clone()}
    for t in range(TD - 1, -1, -1):
        e = unet(x, torch.full((len(x),), t))
        x = (x - bD[t] / (1 - abD[t]).sqrt() * e) / aD[t].sqrt()
        if t > 0: x = x + bD[t].sqrt() * torch.randn_like(x)
        if t in (150, 100, 50, 0): traj[t] = x[0, 0].clone()
gen_d = ((x.clamp(-1, 1) + 1) / 2).reshape(64, 784)
nums["ddpm_sample_seconds"] = float(time.time() - t0)
fig, axs = plt.subplots(1, 5, figsize=(7.5, 1.7))
for a, t in zip(axs, [TD, 150, 100, 50, 0]):
    a.imshow(traj[t].numpy(), cmap="gray_r"); a.axis("off"); a.set_title("pure noise" if t == TD else ("generated" if t == 0 else f"step {t}"), fontsize=9)
plt.savefig(f"{OUT}/gen_ddpm_trajectory.png", dpi=170, bbox_inches="tight"); plt.show()
show_grid(gen_d.numpy(), 8, 8, f"{OUT}/gen_ddpm_samples.png", size=0.6)
with torch.no_grad():
    probs = F.softmax(clf(gen_d), 1); lab = probs.argmax(1).numpy()
nums["ddpm_digit_frac"] = (np.bincount(lab, minlength=10) / len(lab)).round(3).tolist()
nums["ddpm_clf_confidence"] = float(probs.max(1).values.mean())
print(f"64 digits generated in {nums['ddpm_sample_seconds']:.0f} s; digits assigned by the classifier:", np.bincount(lab, minlength=10),
      f"| mean classifier confidence {nums['ddpm_clf_confidence']:.2f}")

sqrt(abar_T) = 0.00551: the last noisy image contains no trace of the digit
loaded trained weights from results/ddpm_mnist.pt


/tmp/ipykernel_3869297/2140449776.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{OUT}/gen_ddpm_trajectory.png", dpi=170, bbox_inches="tight"); plt.show()


64 digits generated in 21 s; digits assigned by the classifier: [ 2  8 11  5  5  7  2 11  7  6] | mean classifier confidence 0.80


/tmp/ipykernel_3869297/2771021278.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(path, dpi=170, bbox_inches="tight"); plt.show()


**Try it.** Use only $T=10$ steps (and a larger final $\beta_T$). How do the generated points change?

## 9. The 2D Ising model: learning the Boltzmann distribution

For spins we **know** the true distribution, $p(\mathbf s\,|\,T)=e^{-H(\mathbf s)/T}/Z$ (Lecture 4), and Metropolis Monte Carlo samples it exactly. Can a network learn it from samples alone?

We train a conditional VAE with the temperature as condition, $\mathbf c=T$, on the Monte Carlo configurations of a $20\times20$ lattice (51 temperatures from 1.0 to 3.5, 200 configurations each; the data set of Lecture 4, regenerated here if missing). Spins $s_i=\pm1$ are mapped to pixels $(s_i+1)/2\in\{0,1\}$, so the same BCE loss applies. To generate at a chosen $T$ we draw $\mathbf z\sim\mathcal N(0,I)$, decode, and draw each spin up with the decoded probability.

We then check the generated configurations **as physicists**: do they have the right magnetisation $\langle|M|\rangle$ and energy per spin at each temperature?

In [23]:
path = "data/ising_Nx20_Ny20.npz"
if not os.path.exists(path):
    path = "data/ising_Nx20_Ny20.npz"
    if not os.path.exists(path):                              # regenerate (checkerboard Metropolis, about a minute)
        r = np.random.default_rng(0); L = 20; Ts = np.round(np.arange(1.0, 3.5 + 1e-9, 0.05), 2)
        y_, x_ = np.indices((L, L)); masks = ((x_ + y_) % 2 == 0, (x_ + y_) % 2 == 1)
        beta = (1 / Ts.astype(np.float32))[None, :, None, None]; s = np.ones((20, len(Ts), L, L), np.float32)
        def sweep(s):
            for m in masks:
                nb = np.roll(s, 1, -1) + np.roll(s, -1, -1) + np.roll(s, 1, -2) + np.roll(s, -1, -2)
                s[...] = np.where(m & (r.random(s.shape, dtype=np.float32) < np.exp(-beta * 2 * s * nb)), -s, s)
        for _ in range(1200): sweep(s)
        st = []
        for k in range(10):
            for _ in range(20): sweep(s)
            st.append(s.copy())
        cf = np.stack(st).transpose(2, 0, 1, 3, 4).reshape(len(Ts), 200, L, L).astype(np.int8)
        os.makedirs("data", exist_ok=True); np.savez_compressed(path, configs=cf, T=Ts)
d = np.load(path); cfg, Ts = d["configs"].astype(np.float32), d["T"]
L = cfg.shape[-1]; N = L * L
print("Monte Carlo data:", cfg.shape, "temperatures", Ts[0], "...", Ts[-1])
Xs = torch.tensor(((cfg + 1) / 2).reshape(-1, N))
Tn = lambda T: (np.asarray(T, np.float32) - 2.25) / 0.75          # the condition: T, rescaled to about [-1.7, 1.7]
Cs = torch.tensor(np.repeat(Tn(Ts), cfg.shape[1]))[:, None]

rng = seed(6)
ivae = VAE(D=N, d=8, cond=1, hidden=(256, 128)); hi_ = train_vae(ivae, Xs, Cs, epochs=60); ivae.eval()
print(f"loss per configuration {hi_[-1, 0]:.1f} = reconstruction {hi_[-1, 1]:.1f} + KL {hi_[-1, 2]:.1f}")

def energy(s):        # energy per spin, J = 1, periodic boundaries; s: (..., L, L) with values +-1
    return -(s * (np.roll(s, 1, -1) + np.roll(s, 1, -2))).mean((-1, -2))
gen_rng = torch.Generator().manual_seed(12)
with torch.no_grad():
    z = torch.randn(len(Ts) * 200, 8, generator=gen_rng)
    prob = torch.sigmoid(ivae.decode(z, Cs))
    gen = (torch.bernoulli(prob, generator=gen_rng) * 2 - 1).numpy().reshape(len(Ts), 200, L, L)
M_mc, M_gen = np.abs(cfg.mean((-1, -2))).mean(1), np.abs(gen.mean((-1, -2))).mean(1)
E_mc, E_gen = energy(cfg).mean(1), energy(gen).mean(1)
nums["ising_cvae_M_maxdiff"] = float(np.abs(M_mc - M_gen).max())
nums["ising_cvae_E_maxdiff"] = float(np.abs(E_mc - E_gen).max())
nums["ising_cvae_E_maxdiff_T"] = float(Ts[np.argmax(np.abs(E_mc - E_gen))])

fig, axs = plt.subplots(1, 2, figsize=(9.5, 3.3))
axs[0].plot(Ts, M_mc, "o-", ms=3, color=blue, label="Monte Carlo"); axs[0].plot(Ts, M_gen, "s--", ms=3, color=orange, label="generated by the CVAE")
axs[1].plot(Ts, E_mc, "o-", ms=3, color=blue, label="Monte Carlo"); axs[1].plot(Ts, E_gen, "s--", ms=3, color=orange, label="generated by the CVAE")
axs[0].set_ylabel(r"$\langle |M| \rangle$"); axs[1].set_ylabel("energy per spin")
TC = 2 / np.log(1 + np.sqrt(2))
for a in axs: a.axvline(TC, color="k", ls=":", lw=1); a.set_xlabel("$T$"); a.legend(frameon=False, fontsize=8); clean(a)
plt.tight_layout(); plt.savefig(f"{OUT}/gen_ising_cvae_observables.png", dpi=170, bbox_inches="tight"); plt.show()

T_show = [1.5, 2.0, 2.25, 2.5, 3.0]
fig, axs = plt.subplots(2, len(T_show), figsize=(8, 3.6))
for k, T in enumerate(T_show):
    i = int(np.argmin(np.abs(Ts - T)))
    for r_, src in enumerate([cfg[i, 0], gen[i, 0]]):
        axs[r_, k].imshow(src, cmap="gray", vmin=-1, vmax=1, interpolation="nearest"); axs[r_, k].set_xticks([]); axs[r_, k].set_yticks([])
    axs[0, k].set_title(f"T = {Ts[i]:.2f}", fontsize=10)
axs[0, 0].set_ylabel("Monte Carlo", fontsize=9); axs[1, 0].set_ylabel("generated", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/gen_ising_cvae_samples.png", dpi=170, bbox_inches="tight"); plt.show()
print({k: round(nums[k], 3) for k in ["ising_cvae_M_maxdiff", "ising_cvae_E_maxdiff", "ising_cvae_E_maxdiff_T"]})

Monte Carlo data: (51, 200, 20, 20) temperatures 1.0 ... 3.5


loss per configuration 139.2 = reconstruction 132.8 + KL 6.3


/tmp/ipykernel_3869297/3391792281.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_ising_cvae_observables.png", dpi=170, bbox_inches="tight"); plt.show()


{'ising_cvae_M_maxdiff': 0.234, 'ising_cvae_E_maxdiff': 0.634, 'ising_cvae_E_maxdiff_T': 2.3}


/tmp/ipykernel_3869297/3391792281.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"{OUT}/gen_ising_cvae_samples.png", dpi=170, bbox_inches="tight"); plt.show()


Compare the curves, not only the pictures. The magnetisation tells whether the model learned the **global** order at each temperature; the energy counts aligned **neighbours**, i.e. the local correlations. Here the spins are drawn independently once the decoder has given their probabilities, so any correlation between neighbours must come through the latent vector $\mathbf z$; where the energy of the generated configurations deviates from Monte Carlo, the model has not captured those correlations.

**Try it.** Generate at $T=2.27$, a temperature between two of the training temperatures. Do the generated configurations interpolate sensibly? Increase the latent dimension $d$: does the energy improve?

## Numbers used on the slides

In [24]:
json.dump(nums, open(f"{RES}/numbers_l6.json", "w"), indent=1)
print(json.dumps(nums, indent=1))

{
 "gauss_P_within_1sigma": 0.6826894921370859,
 "gauss_peak_sigma02": 1.9947114020071635,
 "mix_mean_estimates": {
  "10": -1.142,
  "100": -0.591,
  "10000": -0.282
 },
 "post_mean": 0.3603117099594829,
 "post_std": 0.24452919880531934,
 "kl_mu_min": -0.25,
 "kl_ml_mean": -0.21149227291354722,
 "kl_ml_std": 1.9291983378841506,
 "kl_p_qstar": 0.26199624527381676,
 "kl_qstar_p": 0.4540578229425971,
 "coin_p7_given_half": 0.1171875,
 "coin_L_at_07": 0.2668279319999998,
 "coin_L_integral": 0.09090909094815097,
 "mle_loglik_1": -4.61345417507708,
 "mle_loglik_3": -3.626732964448672,
 "vae16_loss": 99.90340289713542,
 "vae16_rec": 80.27467460530599,
 "vae16_kl": 19.628728348795573,
 "vae_nn_dist_new": 4.72645378112793,
 "vae_nn_dist_test": 4.0293474197387695,
 "clf_acc": 0.9707,
 "cvae_agreement": [
  0.975,
  0.935,
  0.985,
  0.98,
  0.965,
  0.94,
  0.955,
  0.96,
  0.965,
  0.965
 ],
 "blood_clf_test_acc": 0.8380590470622625,
 "blood_cvae_class_agreement": [
  0.04,
  0.81,
  0.98,
  0